# Lesson 27 Lab — Production Deployment, Versioning, and Rollback

**Puzzle:** What makes a quantized release safely reversible?

This notebook keeps the RTX 5090 outputs from a complete run. Read the theory cells, make a prediction, and then use **Run All** on your own GPU.


## Why this matters

A quantized artifact is not ready when conversion finishes; it is ready when a versioned candidate passes frozen gates and a tested rollback path exists. Release decisions should be deterministic from evidence, so the same manifest produces the same promote-or-rollback result rather than depending on operator optimism.


## 0. Predict before running

1. Predict whether the candidate passes a 10% latency gate and an RMSE≤0.5 gate.
2. Explain why both gates are required even when latency improves.
3. List the additional evidence needed before changing `rollback` to a live canary.

For each answer, name the observation that would prove you wrong.


## 1. Name the concrete objects

A release unit includes immutable model/tokenizer/recipe/runtime/container identities, metrics, canary policy, observability, and an already verified rollback target.

- Model, tokenizer, quantization recipe, runtime, and GPU compatibility form one release unit.
- Canary gates need quality, latency, error-rate, and capacity thresholds.
- Rollback must reference an already verified immutable baseline.


## 2. Derive the mechanism

Promotion is a state machine: offline gates -> load/smoke -> shadow -> canary -> broader rollout. Every transition consumes fixed evidence and has an automatic stop/rollback condition.

A release manifest binds candidate and baseline revisions, environment, quantization recipe, quality thresholds, performance SLOs, owners, observability, canary fraction, and rollback target. Each gate evaluates a named artifact; the decision is the conjunction for critical gates, not an average score.

Rollback must restore a loadable, compatible baseline and be rehearsed before promotion. A local synthetic decision can validate the gate machinery while remaining explicit that no container, traffic, or service health signal was exercised.

### Mechanism at a glance

```mermaid
stateDiagram-v2
  [*] --> Offline
  Offline --> LoadSmoke: quality and performance pass
  Offline --> Rollback: gate fails
  LoadSmoke --> Shadow: load and compatibility pass
  LoadSmoke --> Rollback: gate fails
  Shadow --> Canary: shadow checks pass
  Shadow --> Rollback: drift or error
  Canary --> Rollout: SLO and quality pass
  Canary --> Rollback: threshold breached
  Rollout --> Rollback: production regression
```

### Walk it step by step

1. **Bind immutable artifacts.** Model, tokenizer, recipe, runtime, container, and GPU compatibility form one release unit.
2. **Pass offline gates.** Quality, numerical, load, and performance checks run before any traffic exposure.
3. **Increase exposure in stages.** Shadow and canary stages consume predeclared health and quality thresholds.
4. **Make rollback executable.** Every stage points to a load-tested baseline and has an automatic or operator-triggered stop condition.


## 3. Verify the execution environment

The next cell asserts CUDA availability, fixes the seed, locates the lesson, and prints a sanitized GPU/PyTorch/CUDA record. Check it before interpreting output.


In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "27-production-rollout"
device = require_cuda()
torch.manual_seed(2026 + 27)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 4. Freeze the comparison

| Role | This run |
|---|---|
| Baseline | versioned BF16 matrix path `bf16-v1` |
| Candidate | reference INT4-dequantized path `reference-int4-v1` |
| Held constant | same tensors, fifteen timing samples, fixed RMSE/latency thresholds |
| Measurements | baseline/candidate median and p90, output error, individual gate booleans, release decision |
| Evidence | `capacity-model` |

**Experiment:** Evaluate a synthetic candidate against frozen gates and emit a release decision plus rollback manifest from measured CUDA output error and timing.


## 5. Read the experiment code

The notebook converts measured CUDA error and timing into a deterministic synthetic release decision and rollback manifest, without claiming live traffic.

The notebook measures both paths, computes error, evaluates two predeclared booleans, and writes a manifest whose decision is `promote_to_canary` only if all gates pass. The rollback target is stored even when the candidate fails.

This is a deterministic release-policy test. It is not a container build, model-card audit, shadow deployment, or canary against live traffic.

Only after these variables match the protocol should the cell be executed.


In [2]:
w=torch.randn(2048,2048,device=device,dtype=torch.bfloat16); x=torch.randn(16,2048,device=device,dtype=torch.bfloat16); ref=x@w.t(); _,_,dq=symmetric_quantize(w,bits=4,group_size=128); dq=dq.bfloat16(); cand=x@dq.t()
base_t=cuda_benchmark(lambda:x@w.t(),warmup=4,repeats=15); cand_t=cuda_benchmark(lambda:x@dq.t(),warmup=4,repeats=15); err=error_metrics(ref,cand)
gates={"rmse_lte_0_5":err["rmse"]<=0.5,"latency_regression_lte_10pct":cand_t["median_ms"]<=base_t["median_ms"]*1.10}
decision="promote_to_canary" if all(gates.values()) else "rollback"
manifest={"candidate":"reference-int4-v1","baseline":"bf16-v1","decision":decision,"gates":gates,"rollback_target":"bf16-v1"}
result=base_result(27,"capacity-model"); result.update({"baseline_timing":base_t,"candidate_timing":cand_t,"output_error":err,"release_manifest":manifest,
    "conclusion":"Frozen synthetic gates produced a deterministic release or rollback decision; no live service canary was claimed."})


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| Baseline median | 0.019360 ms |
| Candidate median | 0.018848 ms |
| Candidate RMSE | 5.317538 |
| Latency gate | yes |
| Quality gate | no |
| Decision | rollback |


## 7. Interpret rather than merely print

Candidate median latency was 0.018848 ms versus 0.019360 ms for the baseline, so the ≤10% regression gate passed. But output RMSE was 5.317538, far above the 0.5 threshold, so the quality gate failed and the manifest selected `rollback`.

A small speed improvement cannot compensate for a failed critical quality gate. The result illustrates why release criteria must be conjunctive and frozen before the candidate is observed.

**Inspection rule:** The manifest is a deployment-control exercise, not evidence that a real service was canaried.


## 8. Keep the evidence label honest

This run is labeled **`capacity-model`**. The calculation uses live GPU information and/or a CUDA probe, but it remains a planning model until a named full engine, quality suite, and service workload execute.

The next cell writes the complete structured result; its existing saved output is part of the checked-in evidence.


In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "baseline_timing": {
    "median_ms": 0.01936,
    "p90_ms": 0.024256,
    "repeats": 15,
    "samples_ms": [
      0.035968,
      0.021824,
      0.020256,
      0.019616,
      0.019168,
      0.019808,
      0.024448,
      0.018944,
      0.01936,
      0.024256,
      0.019168,
      0.018912,
      0.018784,
      0.019072,
      0.018848
    ],
    "warmup": 4
  },
  "candidate_timing": {
    "median_ms": 0.018848,
    "p90_ms": 0.019424,
    "repeats": 15,
    "samples_ms": [
      0.020192,
      0.019072,
      0.018848,
      0.019008,
      0.01904,
      0.018656,
      0.018528,
      0.019424,
      0.019776,
      0.018272,
      0.01856,
      0.018528,
      0.018432,
      0.018944,
      0.018784
    ],
    "warmup": 4
  },
  "conclusion": "Frozen synthetic gates produced a deterministic release or rollback decision; no live service canary was claimed.",
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce 

## 9. Make the bounded decision

> Automate the decision and rollback metadata before exposing traffic; never improvise rollback after a regression.

**Acceptance/rollback:** Version every artifact, define quality/latency/error/capacity thresholds, monitor slices, and test the rollback command before canary traffic.

**Failure analysis:** Changing thresholds after seeing the result converts a gate into a justification. A rollback identifier without a verified artifact is not a rollback plan. Production promotion also needs sustained load, error rates, GPU health, output monitoring, and a human/operator decision path.


## 10. Extend the evidence

Package baseline and candidate into pinned containers, validate cold load and warm restart, run an offline quality suite and shadow traffic, then perform a small canary with automated rollback triggers. Rehearse the rollback and record recovery time before expanding traffic.

The full derivation, reproduction command, evidence boundary and primary references are in [`README.md`](README.md).
